# 🔍 Detecção de Anomalias em Transações Financeiras

**Autor:** Cristian Yela  
**Data:** 2026  
**Ambiente:** Google Colab / Jupyter Notebook

---

## 📋 Sumário

1. [Configuração do Ambiente](#1-configuração-do-ambiente)
2. [Geração de Dados Sintéticos](#2-geração-de-dados-sintéticos)
3. [Análise Exploratória (EDA)](#3-análise-exploratória-eda)
4. [Pré-processamento](#4-pré-processamento)
5. [Balanceamento de Classes](#5-balanceamento-de-classes)
6. [Modelos de Detecção](#6-modelos-de-detecção)
7. [Avaliação Comparativa](#7-avaliação-comparativa)
8. [Explicabilidade com SHAP](#8-explicabilidade-com-shap)
9. [Conclusões](#9-conclusões)

---

> **⚠️ Nota:** Este projeto utiliza dados **sintéticos** gerados aleatoriamente para fins educacionais e de portfólio. Em produção, substitua por dados reais anonimizados.


## 1. Configuração do Ambiente

Instalação das bibliotecas necessárias. No Google Colab, execute a célula abaixo:


In [ ]:
# ============================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================
# Descomente a linha abaixo se estiver no Colab e precisar instalar

# !pip install -q imbalanced-learn xgboost shap tensorflow plotly

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve,
    f1_score, precision_score, recall_score
)
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
import shap
import tensorflow as tf
from tensorflow.keras import layers, Model, Sequential
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configurações visuais
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Todas as bibliotecas importadas com sucesso!")
print(f"TensorFlow versão: {tf.__version__}")
print(f"NumPy versão: {np.__version__}")
print(f"Pandas versão: {pd.__version__}")


## 2. Geração de Dados Sintéticos

Vamos criar um dataset de **10.000 transações** com **20 features**, simulando um cenário realista de fraude bancária onde apenas **~1% das transações são fraudulentas**.

### Características simuladas:
- `valor_transacao`: valor em reais
- `hora_dia`: horário da transação (0-23)
- `dia_semana`: dia da semana
- `idade_conta_dias`: há quanto tempo a conta existe
- `num_transacoes_24h`: transações nas últimas 24h
- `num_transacoes_7d`: transações nos últimos 7 dias
- `valor_medio_30d`: valor médio de transações no último mês
- `desvio_valor_30d`: desvio padrão dos valores
- `pais_risco`: score de risco do país
- `vpn_detectado`: uso de VPN (binário)
- `dispositivo_novo`: dispositivo nunca usado antes
- `tentativas_senha`: tentativas de senha incorretas
- `tempo_sessao_seg`: duração da sessão
- `latencia_ms`: latência da rede
- `score_credito`: score de crédito (300-850)
- `renda_estimada`: renda estimada do cliente
- `divida_total`: dívida total
- `num_cartoes`: número de cartões vinculados
- `transacao_internacional`: flag de transação internacional
- `merchant_risco`: score de risco do estabelecimento


In [ ]:
# ============================================================
# 2. GERAÇÃO DE DADOS SINTÉTICOS
# ============================================================

np.random.seed(42)

# Parâmetros do dataset
N_SAMPLES = 10000
N_FEATURES = 20
FRAUD_RATIO = 0.01  # 1% de fraudes

# Gerar dados com make_classification
X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[1 - FRAUD_RATIO, FRAUD_RATIO],
    flip_y=0.03,
    class_sep=1.5,
    random_state=42
)

# Nomes das features
feature_names = [
    'valor_transacao', 'hora_dia', 'dia_semana',
    'idade_conta_dias', 'num_transacoes_24h',
    'num_transacoes_7d', 'valor_medio_30d',
    'desvio_valor_30d', 'pais_risco', 'vpn_detectado',
    'dispositivo_novo', 'tentativas_senha',
    'tempo_sessao_seg', 'latencia_ms',
    'score_credito', 'renda_estimada',
    'divida_total', 'num_cartoes',
    'transacao_internacional', 'merchant_risco'
]

# Criar DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['fraude'] = y

# Ajustar escalas para valores realistas
df['valor_transacao'] = np.abs(df['valor_transacao']) * 800 + 15
df['hora_dia'] = np.clip((df['hora_dia'] + 3) * 4, 0, 23).astype(int)
df['dia_semana'] = np.clip((df['dia_semana'] + 3.5) * 1, 0, 6).astype(int)
df['idade_conta_dias'] = np.abs(df['idade_conta_dias']) * 400 + 30
df['num_transacoes_24h'] = np.clip(np.abs(df['num_transacoes_24h']) * 5, 0, 50).astype(int)
df['num_transacoes_7d'] = np.clip(np.abs(df['num_transacoes_7d']) * 15, 0, 200).astype(int)
df['valor_medio_30d'] = np.abs(df['valor_medio_30d']) * 600 + 20
df['desvio_valor_30d'] = np.abs(df['desvio_valor_30d']) * 300 + 5
df['pais_risco'] = np.clip((df['pais_risco'] + 2) * 25, 0, 100)
df['vpn_detectado'] = (df['vpn_detectado'] > 0.5).astype(int)
df['dispositivo_novo'] = (df['dispositivo_novo'] > 0.7).astype(int)
df['tentativas_senha'] = np.clip(np.abs(df['tentativas_senha']) * 3, 0, 10).astype(int)
df['tempo_sessao_seg'] = np.abs(df['tempo_sessao_seg']) * 300 + 30
df['latencia_ms'] = np.abs(df['latencia_ms']) * 100 + 20
df['score_credito'] = np.clip((df['score_credito'] + 2) * 140 + 300, 300, 850).astype(int)
df['renda_estimada'] = np.abs(df['renda_estimada']) * 5000 + 1500
df['divida_total'] = np.abs(df['divida_total']) * 10000 + 500
df['num_cartoes'] = np.clip(np.abs(df['num_cartoes']) * 3 + 1, 1, 10).astype(int)
df['transacao_internacional'] = (df['transacao_internacional'] > 0.3).astype(int)
df['merchant_risco'] = np.clip((df['merchant_risco'] + 2) * 25, 0, 100)

# Adicionar alguns outliers artificiais nas fraudes para tornar mais realista
fraud_indices = df[df['fraude'] == 1].index
df.loc[fraud_indices, 'valor_transacao'] *= np.random.uniform(1.5, 5.0, size=len(fraud_indices))
df.loc[fraud_indices, 'tentativas_senha'] += np.random.randint(1, 5, size=len(fraud_indices))
df.loc[fraud_indices, 'tempo_sessao_seg'] *= np.random.uniform(0.1, 0.5, size=len(fraud_indices))

print("=" * 60)
print("📊 RESUMO DO DATASET")
print("=" * 60)
print(f"Total de transações: {len(df):,}")
print(f"Transações normais:  {(df['fraude'] == 0).sum():,} ({(df['fraude'] == 0).mean()*100:.2f}%)")
print(f"Transações fraudulentas: {df['fraude'].sum():,} ({df['fraude'].mean()*100:.2f}%)")
print(f"Número de features: {len(feature_names)}")
print("
Primeiras 5 linhas:")
df.head()


## 3. Análise Exploratória (EDA)

Vamos visualizar a distribuição das classes, correlações e padrões que diferenciam transações normais de fraudulentas.


In [ ]:
# ============================================================
# 3. ANÁLISE EXPLORATÓRIA DE DADOS (EDA)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribuição das classes
ax1 = axes[0, 0]
class_counts = df['fraude'].value_counts()
colors = ['#2ecc71', '#e74c3c']
ax1.bar(['Normal (0)', 'Fraude (1)'], class_counts.values, color=colors, edgecolor='black')
ax1.set_ylabel('Quantidade')
ax1.set_title('Distribuição das Classes', fontweight='bold', fontsize=13)
for i, v in enumerate(class_counts.values):
    ax1.text(i, v + 50, f'{v:,}
({v/len(df)*100:.2f}%)', ha='center', fontweight='bold')

# 2. Distribuição do valor da transação por classe
ax2 = axes[0, 1]
df[df['fraude'] == 0]['valor_transacao'].hist(bins=50, alpha=0.7, label='Normal', color='#2ecc71', ax=ax2)
df[df['fraude'] == 1]['valor_transacao'].hist(bins=50, alpha=0.7, label='Fraude', color='#e74c3c', ax=ax2)
ax2.set_xlabel('Valor da Transação (R$)')
ax2.set_ylabel('Frequência')
ax2.set_title('Distribuição do Valor da Transação', fontweight='bold', fontsize=13)
ax2.legend()

# 3. Boxplot: Score de Crédito vs Fraude
ax3 = axes[1, 0]
sns.boxplot(data=df, x='fraude', y='score_credito', ax=ax3, palette=colors)
ax3.set_xticklabels(['Normal', 'Fraude'])
ax3.set_title('Score de Crédito por Classe', fontweight='bold', fontsize=13)

# 4. Heatmap de correlação (top 10 features mais correlacionadas com fraude)
ax4 = axes[1, 1]
corr_with_target = df.corr()['fraude'].abs().sort_values(ascending=False)[1:11]
top_features = list(corr_with_target.index) + ['fraude']
corr_matrix = df[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax4, square=True)
ax4.set_title('Top 10 Correlações com Fraude', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.show()

print("
📈 Correlação com a variável alvo (fraude):")
print(df.corr()['fraude'].abs().sort_values(ascending=False).head(11))


## 4. Pré-processamento

Separação em treino/teste, normalização e preparação dos dados para modelagem.


In [ ]:
# ============================================================
# 4. PRÉ-PROCESSAMENTO
# ============================================================

# Separar features e target
X = df.drop('fraude', axis=1)
y = df['fraude']

# Split estratificado (mantém proporção de fraudes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 60)
print("📦 DIVISÃO TREINO / TESTE")
print("=" * 60)
print(f"Treino: {len(X_train):,} amostras")
print(f"  - Normais: {(y_train == 0).sum():,}")
print(f"  - Fraudes: {(y_train == 1).sum():,}")
print(f"Teste:  {len(X_test):,} amostras")
print(f"  - Normais: {(y_test == 0).sum():,}")
print(f"  - Fraudes: {(y_test == 1).sum():,}")

# Normalização com StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converter de volta para DataFrame (útil para SHAP depois)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names, index=X_test.index)

print("
✅ Dados normalizados com StandardScaler")
print(f"Média do treino (deve ser ~0): {X_train_scaled.mean():.6f}")
print(f"Desvio padrão do treino (deve ser ~1): {X_train_scaled.std():.6f}")


## 5. Balanceamento de Classes

Vamos aplicar 4 técnicas diferentes e comparar seus efeitos:

1. **SMOTE** — oversampling sintético
2. **Random Undersampling** — redução da classe majoritária
3. **SMOTEENN** — SMOTE + limpeza de ruído
4. **Class Weights** — ponderação no modelo (sem alterar dados)


In [ ]:
# ============================================================
# 5. BALANCEAMENTO DE CLASSES
# ============================================================

print("=" * 60)
print("⚖️ TÉCNICAS DE BALANCEAMENTO")
print("=" * 60)

# Original
print(f"
📌 Original:       {np.bincount(y_train)}")

# 1. SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"📌 SMOTE:          {np.bincount(y_train_smote)}")

# 2. Random Undersampling
under = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under.fit_resample(X_train_scaled, y_train)
print(f"📌 Undersampling:  {np.bincount(y_train_under)}")

# 3. SMOTE + Edited Nearest Neighbours
smote_enn = SMOTEENN(random_state=42)
X_train_se, y_train_se = smote_enn.fit_resample(X_train_scaled, y_train)
print(f"📌 SMOTEENN:       {np.bincount(y_train_se)}")

# 4. ADASYN (alternativa ao SMOTE)
adasyn = ADASYN(random_state=42)
X_train_ada, y_train_ada = adasyn.fit_resample(X_train_scaled, y_train)
print(f"📌 ADASYN:         {np.bincount(y_train_ada)}")

# Visualização
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
titles = ['Original', 'SMOTE', 'Undersampling', 'SMOTEENN', 'ADASYN']
datasets = [
    (X_train_scaled, y_train),
    (X_train_smote, y_train_smote),
    (X_train_under, y_train_under),
    (X_train_se, y_train_se),
    (X_train_ada, y_train_ada)
]

for ax, (X_data, y_data), title in zip(axes, datasets, titles):
    counts = np.bincount(y_data)
    ax.bar(['Normal', 'Fraude'], counts, color=['#2ecc71', '#e74c3c'], edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Quantidade')
    for i, v in enumerate(counts):
        ax.text(i, v + max(counts)*0.02, str(v), ha='center', fontweight='bold')

plt.suptitle('Comparação das Técnicas de Balanceamento', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Modelos de Detecção

Vamos treinar 6 modelos diferentes, cobrindo abordagens supervisionadas, não-supervisionadas e híbridas:

| Modelo | Tipo | Descrição |
|--------|------|-----------|
| Logistic Regression | Supervisionado | Baseline simples e interpretável |
| Random Forest | Supervisionado | Ensemble de árvores |
| XGBoost | Supervisionado | Gradient boosting (estado da arte) |
| Isolation Forest | Não-supervisionado | Isola anomalias via árvores |
| Local Outlier Factor | Não-supervisionado | Baseado em densidade local |
| Autoencoder | Não-supervisionado | Rede neural de reconstrução |


In [ ]:
# ============================================================
# 6. MODELOS DE DETECÇÃO
# ============================================================

# Dicionário para armazenar resultados
resultados = {}

print("=" * 60)
print("🤖 TREINAMENTO DOS MODELOS")
print("=" * 60)

# --------------------------------------------------
# 6.1 LOGISTIC REGRESSION (Baseline)
# --------------------------------------------------
print("
🔹 Treinando Logistic Regression...")
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
resultados['Logistic Regression'] = {
    'y_pred': y_pred_lr, 'y_prob': y_prob_lr, 'modelo': lr
}

# --------------------------------------------------
# 6.2 RANDOM FOREST
# --------------------------------------------------
print("🔹 Treinando Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             class_weight='balanced_subsample',
                             random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]
resultados['Random Forest'] = {
    'y_pred': y_pred_rf, 'y_prob': y_prob_rf, 'modelo': rf
}

# --------------------------------------------------
# 6.3 XGBoost (com dados SMOTE)
# --------------------------------------------------
print("🔹 Treinando XGBoost (com SMOTE)...")
# Calcular scale_pos_weight
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_smote, y_train_smote)
y_pred_xgb = xgb.predict(X_test_scaled)
y_prob_xgb = xgb.predict_proba(X_test_scaled)[:, 1]
resultados['XGBoost + SMOTE'] = {
    'y_pred': y_pred_xgb, 'y_prob': y_prob_xgb, 'modelo': xgb
}

# --------------------------------------------------
# 6.4 ISOLATION FOREST (Não-supervisionado)
# --------------------------------------------------
print("🔹 Treinando Isolation Forest...")
iso = IsolationForest(
    contamination=FRAUD_RATIO,
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train_scaled)
# -1 = anomalia, 1 = normal
y_pred_iso = (iso.predict(X_test_scaled) == -1).astype(int)
# Score de anomalia (quanto maior, mais anômalo)
y_score_iso = -iso.score_samples(X_test_scaled)
resultados['Isolation Forest'] = {
    'y_pred': y_pred_iso, 'y_prob': y_score_iso, 'modelo': iso
}

# --------------------------------------------------
# 6.5 LOCAL OUTLIER FACTOR
# --------------------------------------------------
print("🔹 Treinando Local Outlier Factor...")
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=FRAUD_RATIO,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_train_scaled)
y_pred_lof = (lof.predict(X_test_scaled) == -1).astype(int)
y_score_lof = -lof.score_samples(X_test_scaled)
resultados['Local Outlier Factor'] = {
    'y_pred': y_pred_lof, 'y_prob': y_score_lof, 'modelo': lof
}

# --------------------------------------------------
# 6.6 AUTOENCODER (Deep Learning)
# --------------------------------------------------
print("🔹 Treinando Autoencoder...")

input_dim = X_train_scaled.shape[1]
encoding_dim = 8

autoencoder = Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(encoding_dim, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(input_dim, activation='linear')
])

autoencoder.compile(optimizer='adam', loss='mse')

# Treinar APENAS com transações normais
X_train_normal = X_train_scaled[y_train == 0]

history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=80,
    batch_size=256,
    validation_split=0.1,
    verbose=0,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    ]
)

# Reconstruir e calcular erro
reconstructions = autoencoder.predict(X_test_scaled, verbose=0)
mse = np.mean((X_test_scaled - reconstructions) ** 2, axis=1)

# Definir threshold no percentil 99 (top 1% = anomalia)
threshold_ae = np.percentile(mse, 99)
y_pred_ae = (mse > threshold_ae).astype(int)

resultados['Autoencoder'] = {
    'y_pred': y_pred_ae, 'y_prob': mse, 'modelo': autoencoder
}

print("
✅ Todos os modelos treinados com sucesso!")


## 7. Avaliação Comparativa

Vamos comparar todos os modelos usando métricas adequadas para dados desbalanceados:

- **Precision**: De todas as transações marcadas como fraude, quantas realmente são?
- **Recall**: De todas as fraudes reais, quantas foram detectadas?
- **F1-Score**: Média harmônica entre Precision e Recall
- **AUC-ROC**: Área sob a curva ROC
- **AUC-PR (Average Precision)**: Mais confiável que ROC em dados desbalanceados
- **Custo Estimado**: Cada FN custa R$100, cada FP custa R$5


In [ ]:
# ============================================================
# 7. AVALIAÇÃO COMPARATIVA
# ============================================================

def calcular_custo(y_true, y_pred, custo_fn=100, custo_fp=5):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return fn * custo_fn + fp * custo_fp

def avaliar_todos(resultados, y_true):
    print("=" * 90)
    print(f"{'Modelo':<25} {'Precisão':>10} {'Recall':>10} {'F1':>10} {'AUC-ROC':>10} {'AUC-PR':>10} {'Custo(R$)':>10}")
    print("=" * 90)

    metricas = {}
    for nome, res in resultados.items():
        y_pred = res['y_pred']
        y_prob = res['y_prob']

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        # AUC-ROC e AUC-PR
        if y_prob is not None and len(np.unique(y_prob)) > 1:
            auc_roc = roc_auc_score(y_true, y_prob)
            auc_pr = average_precision_score(y_true, y_prob)
        else:
            auc_roc = 0.5
            auc_pr = y_true.mean()

        custo = calcular_custo(y_true, y_pred)

        metricas[nome] = {
            'precision': prec, 'recall': rec, 'f1': f1,
            'auc_roc': auc_roc, 'auc_pr': auc_pr, 'custo': custo
        }

        print(f"{nome:<25} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc_roc:>10.4f} {auc_pr:>10.4f} {custo:>10.0f}")

    print("=" * 90)
    return metricas

metricas = avaliar_todos(resultados, y_test)


In [ ]:
# ============================================================
# 7.1 VISUALIZAÇÃO DAS MÉTRICAS
# ============================================================

# DataFrame de comparação
df_metricas = pd.DataFrame(metricas).T
df_metricas = df_metricas.sort_values('auc_pr', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. AUC-PR comparativo
ax1 = axes[0, 0]
df_metricas['auc_pr'].plot(kind='barh', ax=ax1, color='steelblue', edgecolor='black')
ax1.set_xlabel('AUC-PR (Average Precision)')
ax1.set_title('AUC-PR por Modelo', fontweight='bold', fontsize=13)
ax1.set_xlim(0, 1)
for i, v in enumerate(df_metricas['auc_pr']):
    ax1.text(v + 0.01, i, f'{v:.3f}', va='center', fontweight='bold')

# 2. Precision vs Recall
ax2 = axes[0, 1]
ax2.scatter(df_metricas['recall'], df_metricas['precision'], s=200, c='coral', edgecolors='black', zorder=3)
for nome, row in df_metricas.iterrows():
    ax2.annotate(nome, (row['recall'], row['precision']),
                 textcoords="offset points", xytext=(8, 5), fontsize=9)
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision vs Recall', fontweight='bold', fontsize=13)
ax2.set_xlim(-0.05, 1.05)
ax2.set_ylim(-0.05, 1.05)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

# 3. Custo estimado
ax3 = axes[1, 0]
df_metricas['custo'].plot(kind='barh', ax=ax3, color='crimson', edgecolor='black')
ax3.set_xlabel('Custo Estimado (R$)')
ax3.set_title('Custo de Erros por Modelo', fontweight='bold', fontsize=13)
for i, v in enumerate(df_metricas['custo']):
    ax3.text(v + max(df_metricas['custo'])*0.01, i, f'R${v:,.0f}', va='center', fontweight='bold')

# 4. Curvas Precision-Recall
ax4 = axes[1, 1]
for nome, res in resultados.items():
    y_prob = res['y_prob']
    if y_prob is not None and len(np.unique(y_prob)) > 1:
        precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_prob)
        ax4.plot(recall_curve, precision_curve, label=nome, linewidth=2)

ax4.set_xlabel('Recall')
ax4.set_ylabel('Precision')
ax4.set_title('Curvas Precision-Recall', fontweight='bold', fontsize=13)
ax4.legend(loc='lower left', fontsize=9)
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7.2 MATRIZ DE CONFUSÃO DO MELHOR MODELO
# ============================================================

# Identificar melhor modelo pelo AUC-PR
melhor_modelo = df_metricas.index[0]
print(f"🏆 Melhor modelo: {melhor_modelo} (AUC-PR: {df_metricas.loc[melhor_modelo, 'auc_pr']:.4f})")

# Matriz de confusão
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Melhor modelo
y_pred_best = resultados[melhor_modelo]['y_pred']
cm_best = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Fraude'], yticklabels=['Normal', 'Fraude'])
axes[0].set_title(f'Matriz de Confusão: {melhor_modelo}', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predito')

# XGBoost (supervisionado)
y_pred_xgb_cm = resultados['XGBoost + SMOTE']['y_pred']
cm_xgb = confusion_matrix(y_test, y_pred_xgb_cm)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Normal', 'Fraude'], yticklabels=['Normal', 'Fraude'])
axes[1].set_title('Matriz de Confusão: XGBoost + SMOTE', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predito')

plt.tight_layout()
plt.show()

# Report detalhado do melhor modelo
print(f"
📋 Classification Report - {melhor_modelo}:")
print(classification_report(y_test, y_pred_best, target_names=['Normal', 'Fraude']))


## 8. Explicabilidade com SHAP

O SHAP (SHapley Additive exPlanations) permite entender **por que** o modelo classificou uma transação como fraude. Isso é essencial para:
- Auditoria e compliance
- Aprovação de modelos em produção
- Feedback para analistas de fraude


In [ ]:
# ============================================================
# 8. EXPLICABILIDADE COM SHAP
# ============================================================

# Usar XGBoost para explicabilidade (melhor interpretação com TreeExplainer)
print("🌳 Gerando explicações SHAP para o XGBoost...")

# TreeExplainer é mais rápido e preciso para modelos baseados em árvore
explainer = shap.TreeExplainer(xgb)

# Calcular SHAP values para uma amostra do teste (500 amostras para performance)
sample_size = min(500, len(X_test_scaled))
X_sample = X_test_scaled[:sample_size]
X_sample_df = X_test.iloc[:sample_size]

shap_values = explainer.shap_values(X_sample)

print(f"✅ SHAP values calculados para {sample_size} transações")
print(f"Shape dos SHAP values: {np.array(shap_values).shape}")


In [ ]:
# ============================================================
# 8.1 SUMMARY PLOT (Importância Global)
# ============================================================

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_names,
    show=False,
    plot_size=(10, 8)
)
plt.title('Importância Global das Features (SHAP)', fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("
📊 Top 10 features mais importantes (média do |SHAP value|):")
importance = pd.DataFrame({
    'feature': feature_names,
    'importancia': np.abs(shap_values).mean(axis=0)
}).sort_values('importancia', ascending=False)
print(importance.head(10).to_string(index=False))


In [ ]:
# ============================================================
# 8.2 EXPLICAÇÃO INDIVIDUAL (Waterfall Plot)
# ============================================================

# Selecionar uma transação de fraude para explicar
fraud_indices = np.where(y_test.values[:sample_size] == 1)[0]

if len(fraud_indices) > 0:
    idx = fraud_indices[0]

    print(f"🔍 Explicando transação #{idx} (REAL: FRAUDE)")
    print(f"   Valor da transação: R${X_sample_df.iloc[idx]['valor_transacao']:.2f}")
    print(f"   Score de crédito: {X_sample_df.iloc[idx]['score_credito']:.0f}")
    print(f"   Tentativas de senha: {X_sample_df.iloc[idx]['tentativas_senha']}")
    print(f"   Probabilidade de fraude: {y_prob_xgb[idx]:.4f}")

    # Waterfall plot
    plt.figure(figsize=(12, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[idx],
            base_values=explainer.expected_value,
            data=X_sample[idx],
            feature_names=feature_names
        ),
        show=False
    )
    plt.title(f'Explicação SHAP - Transação {idx}', fontweight='bold', fontsize=13, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma fraude encontrada na amostra para explicar.")


In [ ]:
# ============================================================
# 8.3 FORCE PLOT (Interativo)
# ============================================================

# Force plot para a mesma transação
if len(fraud_indices) > 0:
    idx = fraud_indices[0]

    plt.figure(figsize=(16, 4))
    shap.force_plot(
        explainer.expected_value,
        shap_values[idx],
        X_sample[idx],
        feature_names=feature_names,
        show=False,
        matplotlib=True
    )
    plt.title(f'Force Plot - Transação {idx}', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 8.4 DEPENDENCE PLOT (Relação Feature vs SHAP)
# ============================================================

# Analisar a relação entre 'valor_transacao' e seu impacto no modelo
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    'valor_transacao',
    shap_values,
    X_sample,
    feature_names=feature_names,
    show=False
)
plt.title('Dependência: Valor da Transação vs Impacto SHAP', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


## 9. Conclusões

### 📊 Resumo dos Resultados

O pipeline completo de detecção de anomalias foi implementado com sucesso, cobrindo:

1. **Geração de dados sintéticos** realistas com desbalanceamento de 1%
2. **Análise exploratória** identificando padrões discriminantes
3. **4 técnicas de balanceamento** comparadas quantitativamente
4. **6 modelos** treinados (supervisionados, não-supervisionados e híbridos)
5. **Avaliação robusta** com AUC-PR, custo de erro e matrizes de confusão
6. **Explicabilidade SHAP** para transparência e auditoria

### 🎯 Principais Aprendizados

- **AUC-PR é a métrica mais confiável** para dados altamente desbalanceados
- **XGBoost com SMOTE** tende a ter o melhor desempenho supervisionado
- **Autoencoders** são poderosos para detectar padrões nunca vistos (zero-day fraud)
- **Isolation Forest** é excelente como primeiro filtro em pipelines de produção
- **SHAP é indispensável** para explicar decisões a stakeholders e reguladores

### 🚀 Próximos Passos

- [ ] Substituir dados sintéticos por dataset real (ex: IEEE-CIS Fraud Detection)
- [ ] Implementar pipeline com `sklearn.pipeline` e `GridSearchCV`
- [ ] Criar API com FastAPI para predição em tempo real
- [ ] Adicionar monitoramento de drift de dados em produção
- [ ] Implementar ensemble híbrido: Isolation Forest + XGBoost em cascata


In [ ]:
# ============================================================
# SALVAR RESULTADOS (Opcional)
# ============================================================

# Salvar o modelo XGBoost
# xgb.save_model('modelo_xgboost_fraud.json')

# Salvar o scaler
# import joblib
# joblib.dump(scaler, 'scaler.pkl')

# Salvar métricas
# df_metricas.to_csv('metricas_comparativas.csv')

print("✅ Pipeline concluído! Modelos e métricas podem ser salvos para deploy.")
print("
💡 Dica: Use o botão 'Salvar uma cópia no GitHub' do Colab")
print("   ou exporte como .ipynb e faça upload no seu repositório.")
